In [22]:
from dotenv import load_dotenv
load_dotenv()

from langchain_core.messages import HumanMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    trim_messages,
)
import base64
import io
import os
from pdf2image import convert_from_path
import fitz
import time

llm = ChatGoogleGenerativeAI(
    model = 'gemini-1.5-flash-8b',
    temperature = 0,
    max_tokens = None,
    timeout = 15,
    max_retries = 2,
)

I0000 00:00:1733763726.306981 21372023 check_gcp_environment_no_op.cc:29] ALTS: Platforms other than Linux and Windows are not supported


In [25]:
def prepare_conversation_history(messages, max_tokens=1048576):
    """
    Prepare and trim the conversation history to fit within token limits.
    
    Args:
        conversation_history: List of messages in the conversation
        max_tokens: Maximum tokens allowed
    
    Returns:
        List of trimmed messages
    """
    # Trim messages to fit within the token limit
    trimmed_messages = trim_messages(
        messages,
        strategy="last",
        token_counter=llm,
        max_tokens=max_tokens,
        allow_partial=True,
    )
    print(f"\nTrimmed conversation history to {len(trimmed_messages)} messages from {len(messages)} messages")
    print(f"Total tokens: {llm.get_num_tokens_from_messages(trimmed_messages)}")
    return trimmed_messages

def extract_text_content(file_path: str) -> list[str]:
    """
    Extract text content from each page of a PDF file.
    
    Args:
        file_path: Path to the PDF file.
    
    Returns:
        List of text content for each page.
    """
    text_content = []

    try:
        with fitz.open(file_path) as pdf:
            for page in pdf:
                text_content.append(page.get_text())
    except Exception as e:
        print(f"Error extracting text from {file_path}: {str(e)}")

    return text_content

def encode_image(image):
    """Convert image to base64 string"""
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    image_bytes = buffer.getvalue()
    return base64.b64encode(image_bytes).decode('utf-8')

def process_slides(course, course_title, handwritten=False,num_docs=None, num_slides=None, overwrite=False):
    """Process slides sequentially with context from previous generations"""
    
    output_dir = f'./output/output_{course}'
    
    os.makedirs(output_dir, exist_ok=True)
    notes_dir = f'/Users/ashoksaravanan/Coding/ScribeLec/Server/summary/parse/Notes/Notes_{course}/'
    
    # Filter for PDF files and validate them
    pdf_files = [f for f in os.listdir(notes_dir) if f.lower().endswith('.pdf')]
    if num_docs:
        pdf_files = pdf_files[:num_docs]
    
    responses = []
    
    for pdf_file in pdf_files:
        try:
            pdf_path = os.path.join(notes_dir, pdf_file)
            
            # Validate PDF file
            if not os.path.isfile(pdf_path):
                print(f"Skipping {pdf_file} - not a valid file")
                continue
                
            # Check file size
            if os.path.getsize(pdf_path) == 0:
                print(f"Skipping {pdf_file} - empty file")
                continue
            
            pdf_output_dir = os.path.join(output_dir, pdf_file.replace('.pdf', ''))
            
            try:
                images = convert_from_path(pdf_path, dpi=50)   
                # Initialize text content
                text_content = []
                
                if not handwritten:
                    # Extract text content
                    pdf_content = extract_text_content(pdf_path)
                    print(f"Extracted text from {len(pdf_content)} pages")
                    
                    # Ensure text content matches number of images
                    if len(pdf_content) != len(images):
                        print(f"Warning: Mismatch between images ({len(images)}) and text content ({len(pdf_content)})")
                        # Take the minimum length to avoid index errors
                        min_length = min(len(images), len(pdf_content))
                        images = images[:min_length]
                        pdf_content = pdf_content[:min_length]
                        print(f"Adjusted to process {min_length} slides")
                    
                    text_content = pdf_content
                else:
                    # For handwritten notes, create empty text content
                    text_content = ["" for _ in range(len(images))]
                    
                if num_slides is not None and len(images) > num_slides:
                    images = images[:num_slides]
                    text_content = text_content[:num_slides]
                
                
                # Check if all slides already exist
                all_slides_exist = True
                for page_number in range(1, len(images) + 1):
                    slide_file = os.path.join(pdf_output_dir, f"{page_number}.txt")
                    if not os.path.exists(slide_file) or overwrite:
                        all_slides_exist = False
                        break
                        
                if all_slides_exist and not overwrite:
                    print(f"Skipping {pdf_file} - all slides already processed")
                    continue
                    
                # Create subdirectory for PDF file
                os.makedirs(pdf_output_dir, exist_ok=True)
                
                # Base prompt
                if handwritten:
                    base_prompt = f"Extract exactly what is written on the handwritten notes, in the context of the course: {course_title}. Output the content in LaTeX format, preserving the formatting of the slide. For any figures you see, try to re-create them in LaTeX. Take note of direction of arrows, placement of labels, and other notations. Below the generation, provide a description of what you see: include specific details that would not be known unless you were given the context of the slide."
                else:
                    base_prompt = f"Extract exactly what is written on the lecture notes, in the context of the course: {course_title}. We will provide you with text content from each of the lecture slides as a reference. Output the extracted content in Markdown format, preserving the formatting of the slide. If there are any figures, provide a very detailed description of what you see, taking note of its placement, orientation, and the reason why it is there in the general context of the lecture. Below the generation, provide a description of what you see: include specific details that would not be known unless you were given the context of the slide."
                
                conversation_history = []
                pdf_responses = []
                
                for text, image_file, page_number in zip(text_content, images, range(len(images))):
                    page_number += 1
                    # Check if individual slide file exists
                    slide_file = os.path.join(pdf_output_dir, f"{page_number}.txt")
                    if os.path.exists(slide_file) and not overwrite:
                        print(f"Skipping slide {page_number} - output already exists")
                        # Read existing response to maintain context
                        with open(slide_file, "r") as f:
                            current_response = f.read()
                        pdf_responses.append(current_response)
                        conversation_history.extend([
                            HumanMessage(content=[{"type": "text", "text": "Previous slide content"}]),
                            AIMessage(content=current_response)
                        ])
                        continue
                        
                    additional_prompt = f"This is slide {page_number} of {len(images)} slides. Use the previous slide's generation to help you understand the context of the current slide. Output the slide number at the top of your response for clarity."
                    
                    # Encode current image
                    image_base64 = encode_image(image_file)
                    
                    # Create message with context from previous responses
                    message_content = [
                        {"type": "text", "text": base_prompt + "\n\n" + additional_prompt},
                        {
                            "type": "image_url",
                            "image_url": f"data:image/png;base64,{image_base64}"
                        }
                    ]
                    
                    # if not handwritten, add the extracted text from the slide
                    if not handwritten:
                        if text: # if text is not empty
                            message_content.append({
                                "type": "text",
                                "text": text
                            })
                        
                    # Create message and get response
                    message = HumanMessage(content=message_content)
                    conversation_history.append(message)
                    
                    
                    while True:
                        try:
                            response = llm.generate([conversation_history])
                            current_response = response.generations[0][0].text
                            
                            if current_response == "":
                                raise Exception("Empty response, retrying...")
                            
                            pdf_responses.append(current_response)
                            # replacing conversation history last message with the following
                            conversation_history.pop()
                            conversation_history.extend([
                                HumanMessage(content=[{"type": "text", "text": "Previous slide content"}]),
                                AIMessage(content=current_response)
                            ])
                            
                            print(f"\nProcessing Slide {page_number}:")
                            print(current_response[:200])
                            
                            # Save individual response in file
                            with open(slide_file, "w") as f:
                                f.write(current_response)
                            
                            break  # Exit the loop if successful
                        
                        except Exception as e:
                            if "payload" in str(e).lower():
                                print("Payload too large, trimming conversation history and retrying...")
                                conversation_history = prepare_conversation_history(conversation_history)
                            elif "exhausted" in str(e).lower():
                                print("Exhausted resources, trying again in 5 seconds...")
                                time.sleep(5)
                            else:
                                print(f"Error processing slide {page_number}: {str(e)}")
                                break  # Exit the loop on non-size-related errors
                        
                print(f"Processing {pdf_file} complete.")
                # saving concatenated response in file
                output_file = os.path.join(output_dir, f"{pdf_file.replace('.pdf', '')}.txt")
                with open(output_file, "w") as f:
                    f.write("\n\n".join(pdf_responses))
                
                responses.extend(pdf_responses)
                
            except Exception as e:
                print(f"Error processing PDF {pdf_file}: {str(e)}")
                continue
                
        except Exception as e:
            print(f"Error reading {pdf_file}: {str(e)}")
            continue

    return responses

In [26]:
# CS243 - Takes around an hour to process all the slides. If I have a batch-size parameter, maybe this can be sped up drastically. For now, ok, but make this a python script, that can run overnight (or on a server).

# Process all slides
responses = process_slides('CS243', "AI Basics")
# responses = process_slides('STAT511', "Statistical Methods", num_docs=2)
print(responses)

Extracted text from 56 pages
Skipping 07L2-Vision.pdf - all slides already processed
Extracted text from 39 pages
Skipping 14L1-DepthFS.pdf - all slides already processed
Extracted text from 33 pages
Skipping 05L2-ML Basics.pdf - all slides already processed
Extracted text from 54 pages
Skipping 15L2-SAT.pdf - all slides already processed
Extracted text from 56 pages
Skipping 11L2-svd-pca-recommender.pdf - all slides already processed
Extracted text from 24 pages
Skipping 13L1-Logic.pdf - all slides already processed
Extracted text from 23 pages
Skipping 09L1-cnn.pdf - all slides already processed
Extracted text from 43 pages
Skipping 08L2-Feature Extraction.pdf - all slides already processed
Extracted text from 50 pages
Skipping 02L2-Intro.pdf - all slides already processed
Extracted text from 27 pages
Skipping 10L1-CrossValidation.pdf - all slides already processed
Extracted text from 37 pages
Skipping slide 1 - output already exists
Skipping slide 2 - output already exists
Skipping 

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..



Processing Slide 31:
# Slide 31

**Understand When the Algorithm Makes No Sense**

Or figuring out when you shouldn't trust a model

```
(Image)
```

*A cartoon drawing of a person, likely a woman, is seated at a desk, lo


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..


Exhausted resources, trying again in 5 seconds...

Processing Slide 32:
# Slide 32

**Description:**

This slide discusses a study on using machine learning to predict pneumonia risk and the probability of death (POD) in patients.  The study, funded by Cost-Effective Heal

Processing Slide 33:
# Slide 33

**Description:**

This slide discusses a study by Caruana et al. (2015) on using rule-based systems in healthcare, specifically for predicting pneumonia risk.  The study found a counterint

Processing Slide 34:
# Slide 34

**Detecting and removing bias from models**

Bias is in the data, not the learning algorithm
Models trained on data will lean any biases in the data
* ML for resume processing will learn g
Error processing slide 35: Empty response, retrying...

Processing Slide 36:
# Slide 36

**Follow-up**

NYC's Anti-Bias AI Hiring Law Has Largely Failed So Far

Although other jurisdictions have shown interest in passing laws in a similar vein to LLL 144, lawmakers have been h

Pr

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..



Processing Slide 13:
# 13

**Solving the example**

Ax = b, in our example:

A = (
−3  2
5  −2)
x = (
x
y)
b = (
−2
7 )

A<sup>-1</sup>Ax = x = A<sup>-1</sup>b

A<sup>-1</sup> =
1
2
1
2
5
4
3
4

x =
1
2
1
2
5
4
3
4
(
−2
7
Exhausted resources, trying again in 5 seconds...


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 Resource has been exhausted (e.g. check quota)..



Processing Slide 14:
# 14

**Multiplication**

Matrix Multiplication is a bit more complicated. A matrix multiplied to a vector applies the linear transformation from one vector to the transformed vector.
Multiplying two 

Processing Slide 15:
# 15

**Multiplication Example**

```
(1  2  3) (7  8) = (1*7 + 2*9 + 3*11  1*8 + 2*10 + 3*12) = (58  64)
(4  5  6) (9 10)   (4*7 + 5*9 + 6*11  4*8 + 5*10 + 6*12)   (139 154)
```

However, in matrix m

Processing Slide 16:
# 16

**Transposition**

A fundamental operation in linear algebra for manipulating data

Given A = ( 1  2  3 )
         ( 4  5  6 )

A<sup>T</sup> = ( 1  4 )
         ( 2  5 )
         ( 3  6 )


**D

Processing Slide 17:
# 17

**Perceptrons (Rosenblatt '58)**

```
X1
W1
X2
W2
in(t)
W3
Σ
X3
Wn
wo(t) = θ
out(t)
```

**Perceptron:** By Mayranna (Own work) [CC-BY-SA-3.0 (http://creativecommons.org/licenses/by-sa/3.0)], vi

Processing Slide 18:
# 18

**Parallelization and Matrix Multiplication**

```
Core 1
a11 a12 a13 a14
a21 a2